# 技能5 · Day 6 上机：IMRaD 论文写作 -- 用 Python 拆解真实论文 + 撰写营销研究各部分

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实论文/库）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **arxiv** Python 包下载/解析真实论文，自动提取其 IMRaD 各部分结构
2. 撰写 **Introduction**（漏斗结构：背景 -> 问题 -> 空白 -> 贡献 -> 结构），基于一个营销研究问题
3. 撰写 **Methods**（研究设计/数据收集/分析方法），确保可复现性
4. 用 **statsmodels** 跑统计检验（t 检验/卡方/Cohen's d），把结果写成 APA 格式学术表述
5. 撰写 **Discussion**（发现解读/局限性/未来方向），展示对研究的深度理解
6. 生成 **APA 第 7 版**参考文献列表（真实引用，格式准确）

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：arxiv（lukasschwab/arxiv.py，1.5k★，arXiv API 封装）+ statsmodels（统计检验）。
营销映射：撰写一篇"营销Agent vs 人工策略效果对比"的 IMRaD 论文。


## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> ⚠️ arxiv 包需要网络连接访问 arXiv API。statsmodels/scipy/numpy 离线可用。
> 若无法联网，TODO1 的 solution 提供了预置的论文元数据 fallback。


In [ ]:
# !pip install arxiv statsmodels scipy numpy -q
# arxiv 包需要网络连接；statsmodels/scipy/numpy 离线可用

## 1. 数据集背景与营销映射

**研究主题**：营销Agent vs 人工策略的效果对比（A/B 测试 + 用户访谈）

**真实论文范例**（用于 IMRaD 结构拆解）：

| 论文 | arXiv ID | 用途 |
|------|----------|------|
| ReAct: Synergizing Reasoning and Acting in Language Models | 2210.03629 | Agent 论文的 IMRaD 结构范例（Yao et al., NeurIPS 2022） |
| Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena | 2306.05685 | 评估方法论引用（Zheng et al., NeurIPS 2023） |

**营销 A/B 测试数据**（用于 Results 部分统计分析）：

| 指标 | 对照组（人工） | 实验组（Agent） | 数据来源 |
|------|:-----------:|:-----------:|---------|
| 内容产出效率（篇/天） | ~8 | ~32 | 模拟真实营销团队2个月数据 |
| 内容 CTR（%） | ~2.1 | ~2.8 | 基于行业基准 |
| 用户满意度（1-10） | ~7.2 | ~7.8 | 模拟用户评分 |

> 💡 在真实研究中，这些数据来自你的 A/B 测试平台。本上机用基于行业基准的可复现数据（固定随机种子）。


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
from scipy import stats as sp_stats
from statsmodels.stats.weightstats import ttest_ind

print("依赖导入完成：numpy, scipy, statsmodels")
print("arxiv 包将在 TODO1 中按需导入")

## TODO 1：用 arxiv 包下载/解析真实论文，提取 IMRaD 结构

**目标**：用 `arxiv` Python 包获取 ReAct 论文（arXiv 2210.03629）的元数据，解析其摘要中的 IMRaD 结构。

**IMRaD 结构识别方法**：
- Introduction 句子：通常包含背景、动机、问题陈述（"we propose"、"this paper"）
- Methods 句子：描述方法、数据集、实验设计（"we evaluate"、"our approach"）
- Results 句子：报告发现、数据、性能（"we find"、"results show"、"achieves"）
- Discussion/Conclusion 句子：总结贡献、局限性（"we demonstrate"、"in summary"）


In [ ]:
# 1. 用 arxiv 包下载/解析真实论文，提取 IMRaD 结构

# 获取 ReAct 论文元数据
try:
    import arxiv
    client = arxiv.Client()
    search = arxiv.Search(id_list=["2210.03629"])
    paper = next(client.results(search))
    paper_info = {
        "title": paper.title,
        "authors": [str(a) for a in paper.authors],
        "summary": paper.summary,
        "published": str(paper.published),
        "url": paper.entry_id
    }
    print("成功从 arXiv API 获取 ReAct 论文元数据")
except Exception as e:
    print(f"arXiv API 不可用（{e}），使用预置元数据")
    paper_info = {
        "title": "ReAct: Synergizing Reasoning and Acting in Language Models",
        "authors": ["Shunyu Yao", "Jeffrey Zhao", "Dian Yu", "Nan Du", "Izhak Shafran", "Karthik Narasimhan", "Yuan Cao"],
        "summary": "While large language models (LLMs) demonstrate impressive performance across tasks in language understanding and interactive decision making, their reasoning and acting capacities are largely studied as separate topics. In this paper, we explore the use of LLMs to generate both reasoning traces and task-specific actions in an interleaved manner. We evaluate our approach on diverse benchmarks. Results show that our approach achieves state-of-the-art performance. We demonstrate the effectiveness of our approach.",
        "published": "2022-10-06",
        "url": "https://arxiv.org/abs/2210.03629"
    }

# 解析摘要的 IMRaD 结构
abstract = paper_info["summary"]
sentences = [s.strip() for s in abstract.replace("\n", " ").split(".") if len(s.strip()) > 10]

imrad_keywords = {
    "Introduction": ["we explore", "this paper", "we study", "while", "however", "we propose"],
    "Methods": ["we evaluate", "our approach", "we use", "method", "we generate", "interleaved"],
    "Results": ["results show", "achieves", "we find", "state-of-the-art", "performance"],
    "Discussion": ["we demonstrate", "in summary", "limitations", "effectiveness", "conclusion"]
}

imrad_analysis = {k: [] for k in imrad_keywords}
for sent in sentences:
    sent_lower = sent.lower()
    for section, keywords in imrad_keywords.items():
        if any(kw in sent_lower for kw in keywords):
            imrad_analysis[section].append(sent)
            break
    else:
        imrad_analysis["Introduction"].append(sent)  # 默认归入 Introduction

print(f"\n论文标题: {paper_info['title']}")
print(f"作者: {', '.join(paper_info['authors'][:3])} et al.")
print(f"发表: {paper_info['published']}")
print(f"\nIMRaD 结构分析（基于摘要 {len(sentences)} 句）:")
for section, sents in imrad_analysis.items():
    print(f"  {section}: {len(sents)} 句")
    for s in sents[:2]:
        print(f"    - {s[:80]}...") if len(s) > 80 else print(f"    - {s}")


## 2. IMRaD 详解：Introduction（漏斗结构）

Introduction 遵循"漏斗结构"--从大到小，从宽到窄：

```
领域背景（宽）  "AI原生营销正在重塑企业增长方式..."
    |
具体问题（窄）  "但现有营销Agent系统缺乏有效的效果评估方法论..."
    |
研究空白（更窄）  "目前没有研究系统地探讨多Agent营销系统的评估指标体系..."
    |
本文贡献（最窄）  "本文提出一个基于LangGraph的多Agent营销系统架构..."
    |
论文结构  "本文第2节介绍方法，第3节呈现结果，第4节讨论..."
```

**核心要点**：
1. 研究背景：2-3 段，引用行业报告和学术文献
2. 研究问题：1-2 段，明确指出当前系统的问题
3. 研究空白：1 段，通过文献综述指出前人没做什么
4. 本文贡献：3-4 个 bullet points，清晰声明本文做了什么
5. 论文结构：1 段，概述后续各节内容

> 详见独立教材 § 3.6.2。


## TODO 2：撰写 Introduction

**研究主题**：营销Agent vs 人工策略的效果对比

**要求**：
1. 研究背景：AI原生营销发展趋势（引用 McKinsey 2025 报告风格）
2. 研究问题：现有营销 Agent 系统缺乏标准化效果评估
3. 研究空白：尚无研究系统对比 Agent 与人工在营销内容创作上的效果
4. 本文贡献：3-4 个 bullet points
5. 论文结构：概述后续各节

将撰写的 Introduction 存为 `introduction_text` 变量（多行字符串）。


In [ ]:
# 2. 撰写 Introduction（漏斗结构）
introduction_text = """## 1. Introduction

### 1.1 研究背景
近年来，大语言模型（LLM）的快速发展催生了 Agent 系统在企业营销中的应用。根据 McKinsey (2025) 的报告，超过 60% 的企业正在或计划在营销环节部署 AI Agent，用于内容创作、客户画像分析和营销策略优化。Agent 系统能够自动化完成从市场调研到内容生成的全流程，显著提升营销效率。然而，Agent 系统的工程实践远领先于学术研究，缺乏系统化的效果评估方法论。

### 1.2 研究问题
尽管多 Agent 营销系统在实践中日益普及，但存在三个关键问题：（1）缺乏标准化的架构设计模式，不同企业各自实现，难以横向比较；（2）缺乏系统化的效果评估方法论，现有评估多依赖主观判断；（3）缺乏 Agent 与人工策略的严格对比研究，无法量化 Agent 的真实价值。

### 1.3 研究空白
通过系统文献综述，我们检索了 2023-2026 年发表在主要数据库中的 120 篇文献，发现现有研究主要集中在 Agent 架构设计（如 ReAct, Yao et al., 2022）和通用能力评估（如 AgentBench），但尚无研究系统地对比 Agent 与人工在营销内容创作上的效果差异。

### 1.4 本文贡献
本文的主要贡献如下：
1. **效果对比框架**：设计了 Agent vs 人工的 A/B 测试对比框架，覆盖效率、质量和满意度三个维度
2. **实证验证**：通过 2 个月的真实 A/B 测试（N=400），量化了 Agent 在营销内容创作中的效果
3. **评估方法论**：提出用 LLM-as-a-judge（Zheng et al., 2023）辅助评估写作质量的方法
4. **实践指南**：总结了 Agent 部署的实践启示和人机协作流程设计建议

### 1.5 论文结构
本文第 2 节介绍研究方法（研究设计/数据收集/分析方法），第 3 节呈现实验结果（统计检验/定性分析），第 4 节讨论发现解读/局限性/未来方向，第 5 节总结全文。"""

print(introduction_text)


## 3. IMRaD 详解：Methods（可复现性）

Methods 的核心要求是**可复现性**：别人读完你的 Methods，应该能用同样的方法重复你的研究。

**Methods 五要素**：
1. 研究设计：Design Science Research (DSR) / 混合方法 / 纯定量
2. 系统架构：Agent 系统的架构描述（附图）
3. 数据收集：定量数据（A/B 测试）+ 定性数据（访谈）
4. 评估指标：任务完成率/工具准确率/幻觉率/延迟/成本
5. 数据分析方法：t 检验/Cohen's d/主题分析法

> 详见独立教材 § 3.6.3。


## TODO 3：撰写 Methods

**要求**：
1. 研究设计：采用混合方法设计（定量 A/B 测试 + 定性用户访谈）
2. 数据收集：A/B 测试（N=400，对照组人工 vs 实验组 Agent，2个月）+ 8位营销人员访谈
3. 评估指标：内容产出效率/CTR/用户满意度
4. 数据分析方法：独立样本 t 检验 + Cohen's d 效应量 + 主题分析法（Braun & Clarke, 2006）

将撰写的 Methods 存为 `methods_text` 变量（多行字符串）。


In [ ]:
# 3. 撰写 Methods（可复现性）
methods_text = """## 2. Methods

### 2.1 研究设计
本研究采用混合方法设计（Creswell & Plano Clark, 2018），结合定量 A/B 测试和定性用户访谈。定量部分通过 A/B 测试评估 Agent 系统与传统人工在营销内容创作上的效果差异，定性部分通过半结构化访谈理解营销人员的工作流程变化。研究流程遵循设计科学研究（DSR）框架（Peffers et al., 2007）。

### 2.2 系统架构
Agent 系统采用基于 LangGraph 的有状态图架构，包含四个核心节点：分析 Agent（解读营销 Brief）、策略 Agent（制定内容策略）、内容 Agent（生成文案）、审核节点（合规检查）。系统部署于云端，使用 GPT-4 作为 LLM backbone。

### 2.3 数据收集
定量数据：
- A/B 测试数据：在某电商企业部署系统，收集 2 个月的营销内容创作数据
  - 对照组（人工）：200 条营销内容，由 5 位营销专员创作
  - 实验组（Agent）：200 条营销内容，由 Agent 系统生成
  - 记录指标：内容产出效率（篇/天）、点击率（CTR）、用户满意度评分（1-10）

定性数据：
- 半结构化访谈：采访 8 位营销人员（每次 45 分钟），探讨工作流程变化和质量感知
- 田野笔记：记录系统部署期间的组织变化

### 2.4 评估指标
1. 内容产出效率 = 总产出篇数 / 工作天数
2. 内容 CTR = 点击数 / 曝光数
3. 用户满意度 = 用户评分均值（1-10 量表）
4. 人工修改率 = 需人工修改的内容数 / Agent 总产出数

### 2.5 数据分析方法
定量分析：使用独立样本 t 检验比较两组效率差异，效应量用 Cohen's d 报告（Cohen, 1988）。CTR 差异使用卡方检验。所有检验显著性水平设为 α = .05。
定性分析：使用主题分析法（Braun & Clarke, 2006）对访谈数据进行编码，识别核心主题。"""

print(methods_text)


## 4. IMRaD 详解：Results（数据说话）

Results 的原则是**先描述再解释**。描述你发现了什么，解释放在 Discussion。

**Results 写作要点**：
1. 用表格呈现核心数据（附表标题和标注）
2. 统计检验结果用 APA 格式：t(df) = X.XX, p < .001, d = X.XX
3. 效应量解读：d=0.2 小，d=0.5 中，d=0.8 大（Cohen, 1988）
4. 先描述定量结果，再描述定性结果

**APA 统计报告格式**：
- t 检验：`实验组的内容产出效率显著高于对照组（t(398) = 25.43, p < .001, d = 2.34）`
- 卡方检验：`转化率差异显著（χ²(1, N=400) = 8.32, p < .01, φ = 0.14）`

> 详见独立教材 § 3.6.4。


## TODO 4：撰写 Results（用 statsmodels 跑统计检验）

**要求**：
1. 生成营销 A/B 测试数据（固定随机种子，N=200/组）
   - 对照组（人工）：内容产出效率均值~8，CTR~2.1%，满意度~7.2
   - 实验组（Agent）：内容产出效率均值~32，CTR~2.8%，满意度~7.8
2. 用 `statsmodels.stats.weightstats.ttest_ind` 跑独立样本 t 检验
3. 计算 Cohen's d 效应量
4. 用 `scipy.stats.chi2_contingency` 跑卡方检验（CTR 是否显著提升）
5. 将统计结果写成 APA 格式学术表述，存为 `results_text` 变量

**关键 API**：
- `from statsmodels.stats.weightstats import ttest_ind` -- t 检验
- `from scipy.stats import chi2_contingency` -- 卡方检验
- Cohen's d = (mean1 - mean2) / pooled_std


In [ ]:
# 4. 撰写 Results（用 statsmodels 跑统计检验）
np.random.seed(42)

# 生成 A/B 测试数据（基于行业基准）
control_efficiency = np.random.normal(8.2, 2.5, 200)      # 人工组：~8篇/天
treatment_efficiency = np.random.normal(32.5, 8.0, 200)    # Agent组：~32篇/天

# 独立样本 t 检验
t_stat, p_val, df_val = ttest_ind(treatment_efficiency, control_efficiency)

# Cohen's d 效应量
pooled_std = np.sqrt(
    ((len(control_efficiency) - 1) * control_efficiency.std()**2 +
     (len(treatment_efficiency) - 1) * treatment_efficiency.std()**2)
    / (len(control_efficiency) + len(treatment_efficiency) - 2)
)
cohen_d = (treatment_efficiency.mean() - control_efficiency.mean()) / pooled_std

# 卡方检验（CTR: 对照组 2.1% vs 实验组 2.8%, N=10000/组）
# 构建列联表: [点击数, 未点击数]
ctrl_clicked = int(0.021 * 10000)  # 210
ctrl_not_clicked = 10000 - ctrl_clicked
trt_clicked = int(0.028 * 10000)   # 280
trt_not_clicked = 10000 - trt_clicked
contingency_table = np.array([
    [ctrl_clicked, ctrl_not_clicked],
    [trt_clicked, trt_not_clicked]
])
chi2_stat, chi2_p, dof, expected = sp_stats.chi2_contingency(contingency_table)

# 用户满意度数据（用于补充分析）
control_satisfaction = np.random.normal(7.2, 1.2, 200)
treatment_satisfaction = np.random.normal(7.8, 1.0, 200)
t_sat, p_sat, df_sat = ttest_ind(treatment_satisfaction, control_satisfaction)
pooled_std_sat = np.sqrt(
    ((len(control_satisfaction) - 1) * control_satisfaction.std()**2 +
     (len(treatment_satisfaction) - 1) * treatment_satisfaction.std()**2)
    / (len(control_satisfaction) + len(treatment_satisfaction) - 2)
)
d_sat = (treatment_satisfaction.mean() - control_satisfaction.mean()) / pooled_std_sat

# 撰写 Results（APA 格式）
results_text = f"""## 3. Results

### 3.1 系统性能评估
在 200 条测试用例上，Agent 系统的内容产出效率为 {treatment_efficiency.mean():.1f} 篇/天，
对照组（人工）为 {control_efficiency.mean():.1f} 篇/天。

### 3.2 A/B 测试结果
实验组（Agent）的内容产出效率显著高于对照组（人工）（t({df_val:.0f}) = {t_stat:.2f}, p < .001, d = {cohen_d:.2f}），
效应量极大（Cohen's d = {cohen_d:.2f}），表明 Agent 在效率上有显著优势。

在内容 CTR 方面，实验组 CTR（2.8%）显著高于对照组（2.1%）（χ²({dof}, N = 20000) = {chi2_stat:.2f}, p = {chi2_p:.4f}），
差异显著但效应量较小（φ = {np.sqrt(chi2_stat / 20000):.3f}）。

用户满意度方面，实验组（M = {treatment_satisfaction.mean():.2f}, SD = {treatment_satisfaction.std():.2f}）
显著高于对照组（M = {control_satisfaction.mean():.2f}, SD = {control_satisfaction.std():.2f}）
（t({df_sat:.0f}) = {t_sat:.2f}, p < .001, d = {d_sat:.2f}），
效应量为小到中等（Cohen's d = {d_sat:.2f}）。

### 3.3 效应量解读
根据 Cohen (1988) 的标准，d = 0.2 为小效应，d = 0.5 为中等效应，d = 0.8 为大效应。
本研究中效率差异的效应量 d = {cohen_d:.2f} 属于大效应，满意度差异 d = {d_sat:.2f} 属于小到中等效应。
这表明 Agent 在效率提升上有巨大优势，但在质量提升上优势较小。

### 3.4 定性分析结果
对 8 位营销人员的访谈进行主题分析，识别出三个核心主题：
主题1：工作流程重构 -- "我的角色从创作者变成了审核者"
主题2：质量感知变化 -- "AI 生成的内容结构好，但情感共鸣差一点"
主题3：技能焦虑与适应 -- "AI 是工具，不是替代品"
"""

print(f"t 检验: t({df_val:.0f}) = {t_stat:.2f}, p = {p_val:.4e}")
print(f"Cohen's d = {cohen_d:.2f}")
print(f"卡方检验: χ²({dof}) = {chi2_stat:.2f}, p = {chi2_p:.4f}")
print(f"满意度 t 检验: t({df_sat:.0f}) = {t_sat:.2f}, p = {p_sat:.4e}, d = {d_sat:.2f}")
print("\n" + "=" * 60)
print(results_text)


## 5. IMRaD 详解：Discussion（论文的灵魂）

Discussion 是论文的"灵魂"--它展示了你对研究的深度理解。

**Discussion 六要素**：
1. 主要发现解读：不只是重复 Results，要解释"为什么"
2. 理论贡献：扩展了什么理论、提出了什么框架
3. 实践启示：对企业有什么 actionable 的建议
4. 局限性：诚实承认（样本/时间/评估/技术局限）
5. 未来研究方向：2-3 个方向
6. 研究伦理声明：IRB/知情同意/数据脱敏

> 详见独立教材 § 3.6.5。


## TODO 5：撰写 Discussion

**要求**：
1. 主要发现解读：Agent 在效率上显著优于人工（d=2.34），但23.5%的人工修改率表明无法完全替代
2. 理论贡献：扩展 DSR 在 AI 系统设计中的应用
3. 实践启示：Agent 应定位为"辅助工具"而非"替代方案"
4. 局限性：样本局限/时间局限/LLM-as-a-judge 评估偏差/技术时效性
5. 未来研究方向：跨企业验证/长期影响/个性化能力
6. 研究伦理声明

将撰写的 Discussion 存为 `discussion_text` 变量（多行字符串）。


In [ ]:
# 5. 撰写 Discussion
discussion_text = """## 4. Discussion

### 4.1 主要发现解读
本研究的核心发现是：基于 LangGraph 的多 Agent 营销系统在效率上显著优于传统人工（效应量 d = {:.2f}），同时在内容质量上也有提升（CTR 提升 33%，满意度提升 8.3%）。这一发现与 Yao et al. (2022) 的研究一致，他们发现 ReAct 模式能有效协调推理和行动。

然而，23.5% 的人工修改率表明，Agent 系统目前还无法完全替代人工。定性分析揭示了原因：Agent 在"情感共鸣"和"品牌特有调性"方面仍有不足。这指向了 LLM 的一个根本局限：它们擅长模式化的内容生成，但不擅长需要深度品牌理解和情感智能的创作。

### 4.2 理论贡献
1. 扩展了 DSR 框架在 AI 系统设计中的应用，提出 Agent 效果评估的标准化流程
2. 丰富了人机协作理论在营销领域的实证证据
3. 验证了 LLM-as-a-judge（Zheng et al., 2023）在写作质量评估中的可行性

### 4.3 实践启示
1. Agent 系统应定位为"辅助工具"而非"替代方案"，重点设计人机协作流程
2. 评估 Agent 系统不应只看效率，还应关注质量、安全性和用户满意度
3. 部署 Agent 系统需要同步设计内容审核机制，防止幻觉导致的合规风险

### 4.4 局限性
1. 样本局限：A/B 测试仅在一个企业进行，外部效度有限
2. 时间局限：2 个月的观察期可能不足以评估长期影响
3. 评估局限：LLM-as-a-judge 本身可能有评估偏差（位置偏差/冗长偏差/自我偏好）
4. 技术局限：研究基于 2026 年 7 月的 LLM 能力，模型快速迭代可能影响结论时效性

### 4.5 未来研究方向
1. 跨企业、跨行业的多场景验证
2. 长期影响研究（6-12 个月）
3. Agent 系统的个性化能力研究
4. Agent Economy 背景下的商业模式创新

### 4.6 研究伦理声明
本研究已通过机构伦理审查（IRB）。所有参与访谈的人员均已签署知情同意书。用户数据已脱敏处理，符合 GDPR 和中国数据安全法要求。""".format(cohen_d)

print(discussion_text)


## 6. APA 第 7 版引用规范

APA（American Psychological Association）格式是社会科学领域最常用的引用格式。

**正文引用**：
- 单作者：(Smith, 2025)
- 两作者：(Smith & Jones, 2025)
- 三作者及以上：(Smith et al., 2025)
- 直接引用：(Smith, 2025, p. 15)
- 机构作者：(NIST, 2024)

**参考文献列表格式**：
- 期刊论文：Author, A. B., & Author, C. D. (Year). Title. *Journal Name*, Vol(Issue), pages. https://doi.org/xxx
- 会议论文：Author, A. B. et al. (Year). Title. In *Conference Name*.
- arXiv 预印本：Author, A. B. (Year). Title. arXiv preprint arXiv:XXXX.XXXXX.
- 书籍：Author, A. B. (Year). *Title* (ed.). Publisher.

> 详见独立教材 § 3.6.6。


## TODO 6：生成 APA 第 7 版参考文献列表

**要求**：
基于以下真实引用，生成 APA 第 7 版格式的参考文献列表：

1. Yao, S., Zhao, J., Yu, D., Du, N., Shafran, I., Narasimhan, K., & Cao, Y. (2022). ReAct: Synergizing reasoning and acting in language models. arXiv preprint arXiv:2210.03629.
2. Zheng, L., Chiang, W. L., Sheng, Y., et al. (2023). Judging LLM-as-a-judge with MT-Bench and Chatbot Arena. arXiv preprint arXiv:2306.05685.
3. Peffers, K., Tuunanen, T., Rothenberger, M. A., & Chatterjee, S. (2007). A design science research methodology for information systems research. *Journal of Management Information Systems*, 24(3), 45-78.
4. Creswell, J. W., & Plano Clark, V. L. (2018). *Designing and conducting mixed methods research* (3rd ed.). SAGE Publications.
5. Braun, V., & Clarke, V. (2006). Using thematic analysis in psychology. *Qualitative Research in Psychology*, 3(2), 77-101.
6. Cohen, J. (1988). *Statistical power analysis for the behavioral sciences* (2nd ed.). Lawrence Erlbaum Associates.

将参考文献列表存为 `references_text` 变量（多行字符串）。


In [ ]:
# 6. 生成 APA 第 7 版参考文献列表
references_text = """## References

Braun, V., & Clarke, V. (2006). Using thematic analysis in psychology. *Qualitative Research in Psychology*, 3(2), 77-101. https://doi.org/10.1191/1478088706qp063oa

Cohen, J. (1988). *Statistical power analysis for the behavioral sciences* (2nd ed.). Lawrence Erlbaum Associates.

Creswell, J. W., & Plano Clark, V. L. (2018). *Designing and conducting mixed methods research* (3rd ed.). SAGE Publications.

Peffers, K., Tuunanen, T., Rothenberger, M. A., & Chatterjee, S. (2007). A design science research methodology for information systems research. *Journal of Management Information Systems*, 24(3), 45-78. https://doi.org/10.2753/MIS0742-1222240302

Yao, S., Zhao, J., Yu, D., Du, N., Shafran, I., Narasimhan, K., & Cao, Y. (2022). ReAct: Synergizing reasoning and acting in language models. arXiv preprint arXiv:2210.03629. https://arxiv.org/abs/2210.03629

Zheng, L., Chiang, W. L., Sheng, Y., Zhuang, S., Wu, Z., Zhuang, Y., Lin, Z., Li, Z., Li, D., Xing, E. P., Zhang, H., Gonzalez, J. E., & Stoica, I. (2023). Judging LLM-as-a-judge with MT-Bench and Chatbot Arena. arXiv preprint arXiv:2306.05685. https://arxiv.org/abs/2306.05685"""

print(references_text)

# 验证 APA 格式完整性
apa_checks = {
    "作者格式（姓, 名首字母.）": "Yao, S." in references_text and "Zheng, L." in references_text,
    "年份在括号中": "(2022)" in references_text and "(2023)" in references_text,
    "期刊名斜体": "*Journal of Management Information Systems*" in references_text,
    "arXiv ID 正确": "2210.03629" in references_text and "2306.05685" in references_text,
    "DOI 链接": "https://doi.org/" in references_text or "https://arxiv.org/abs/" in references_text,
    "按字母排序": references_text.index("Braun") < references_text.index("Cohen") < references_text.index("Creswell")
}
print("\nAPA 格式检查:")
for check, passed in apa_checks.items():
    print(f"  {'✅' if passed else '❌'} {check}")


## 7. 反思与前沿

### 反思问题
1. 你的 Introduction 是否清晰陈述了研究问题和贡献？能否让非专业人士理解你为什么做这个研究？
2. 你的 Methods 是否足够详细，别人能复现你的研究？哪些细节最容易遗漏？
3. Results 中的统计检验结果是否用 APA 格式正确报告了？效应量大小意味着什么？
4. Discussion 中的局限性是否诚实？有没有"报喜不报忧"的倾向？

### 2026 前沿：LLM-as-a-judge 评估写作质量
在 IMRaD 论文写作中，**LLM-as-a-judge**（NeurIPS 2023, arXiv 2306.05685）可用于：
- **写作质量自检**：让 LLM 扮演"论文审稿人"，按预设 criteria（IMRaD 结构完整性/逻辑连贯性/学术表述规范性）自动评分
- **引用格式校验**：用 LLM 检查参考文献是否符合 APA 第 7 版格式
- **逻辑链审查**：让 LLM 追踪 Introduction -> Methods -> Results -> Discussion 的逻辑一致性

**注意**：LLM-as-a-judge 是辅助工具，有自身偏差（偏好长答案/位置偏差/自我偏好）。它对应因果阶梯 L1（对文本的关联分析），不能替代真实同行评审（L2 干预：修改后重新提交）。定位为"投稿前自检工具"。

参考 [arXiv 2306.05685](https://arxiv.org/abs/2306.05685)（NeurIPS 2023, LLM-as-a-judge）+ [arxiv Python 包](https://github.com/lukasschwab/arxiv.py)。
